# Section 10 — SEEKR: Replay + Selective Distillation (Vast H100)

Tests SEEKR on V-JEPA with the correct **JEPA latent-regression loss** on walkindia-200k.

**Method**: Continual adaptation via:
1. Replay buffer of clips from previous sessions
2. Selective distillation on the top-K most forgettable transformer blocks
   (ranked by L2 drift between frozen session snapshot and current student)

Full encoder trains — no LoRA, no frozen backbone. This is the most expressive
of the three methods and the only one not tested by the collaborator due to OOM.

**Evaluation**: Cycle@K and Overlap@K on held-out val shards.

## 0. Install & clone

In [ ]:
import os

if not os.path.exists('/workspace/vjepa2'):
    os.system('git clone https://github.com/facebookresearch/vjepa2.git /workspace/vjepa2')

os.chdir('/workspace/vjepa2')
os.system('pip install -e ".[dev]" -q')
os.system('pip install -q webdataset huggingface_hub opencv-python-headless')
print('Install complete.')

## 1. Configuration

In [ ]:
import os

# ── HuggingFace token — replace with your token ───────────────────────────────
HF_TOKEN = "YOUR_HF_TOKEN_HERE"

# Set GOPEN_CURL so webdataset's curl subprocess authenticates correctly
os.environ["GOPEN_CURL"] = (
    f'curl -H "Authorization: Bearer {HF_TOKEN}" '
    f'--connect-timeout 30 --retry 3 -f -s -L'
)

# ── Paths ─────────────────────────────────────────────────────────────────────
CKPT_PATH = "/workspace/vjepa2_1_vitb_dist_vitG_384.pt"
CKPT_DIR  = "/workspace/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Dataset ───────────────────────────────────────────────────────────────────
HF_DATASET        = "anonymousML123/walkindia-200k"
SESSION_SHARDS    = [(0, 37), (38, 75), (76, 99)]
VAL_SHARDS        = (100, 102)   # 3 val shards
CLIPS_PER_SESSION = 5000
VAL_CLIPS         = 500

# ── Model ─────────────────────────────────────────────────────────────────────
IMG_SIZE        = 224
FRAMES_PER_CLIP = 16
BATCH_SIZE      = 8    # H100 80GB handles this easily
NUM_EPOCHS      = 3
LR_BACKBONE     = 5e-5
LR_PREDICTOR    = 1e-4
EMA_MOMENTUM    = 0.996
GRAD_CLIP       = 1.0

# ── JEPA masking ──────────────────────────────────────────────────────────────
TARGET_MASK_RATIO = 0.20
NUM_MASK_BLOCKS   = 4

# ── SEEKR ─────────────────────────────────────────────────────────────────────
REPLAY_BUDGET      = 50
SEEKR_TOP_K_BLOCKS = 4
SEEKR_DISTILL_W    = 0.1
SEEKR_REPLAY_W     = 0.5

# ── Evaluation ────────────────────────────────────────────────────────────────
EVAL_K = 10

print("Config ready.")
print(f"GOPEN_CURL set: {os.environ['GOPEN_CURL'][:50]}...")

## 2. Download V-JEPA checkpoint

In [ ]:
import subprocess

if not os.path.exists(CKPT_PATH):
    print("Downloading V-JEPA 2.1 ViT-B checkpoint (~340 MB)...")
    subprocess.run([
        "wget", "-q",
        "https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt",
        "-O", CKPT_PATH
    ], check=True)
    print("Done.")
else:
    print(f"Checkpoint found: {CKPT_PATH}")

## 3. Imports + build encoder

In [ ]:
import sys, torch, copy, time, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import webdataset as wds
from torch.utils.data import DataLoader, IterableDataset

sys.path.insert(0, '/workspace/vjepa2')

try:
    import app.vjepa_2_1.models.vision_transformer as vit_module
except ModuleNotFoundError:
    hub_path = "/root/.cache/torch/hub/facebookresearch_vjepa2_main"
    sys.path.insert(0, hub_path)
    import app.vjepa_2_1.models.vision_transformer as vit_module

import src.datasets.utils.video.transforms as video_transforms
import src.datasets.utils.video.volume_transforms as volume_transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


def build_encoder():
    model = vit_module.vit_base(
        patch_size=16,
        img_size=(IMG_SIZE, IMG_SIZE),
        num_frames=64,
        tubelet_size=2,
        use_sdpa=True,
        use_SiLU=False,
        wide_SiLU=True,
        uniform_power=False,
        use_rope=True,
        img_temporal_dim_size=1,
        interpolate_rope=True,
    )
    ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
    sd   = ckpt["ema_encoder"]
    sd   = {k.replace("module.", "").replace("backbone.", ""): v
            for k, v in sd.items()}
    msg  = model.load_state_dict(sd, strict=True)
    print(f"Encoder loaded: {msg}")
    print(f"embed_dim: {model.embed_dim}  |  blocks: {len(model.blocks)}")
    return model


class JEPAPredictor(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Linear(embed_dim * 2, embed_dim),
        )

    def forward(self, x):
        return self.net(x.mean(dim=1))


print("Imports done.")

## 4. Video transform

In [ ]:
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)


def build_transform(augment=False):
    short = int(256.0 / 224 * IMG_SIZE)
    ops   = [video_transforms.Resize(short, interpolation="bilinear")]
    ops  += [video_transforms.RandomCrop((IMG_SIZE, IMG_SIZE))
              if augment else
              video_transforms.CenterCrop((IMG_SIZE, IMG_SIZE))]
    ops  += [volume_transforms.ClipToTensor(),
              video_transforms.Normalize(mean=MEAN, std=STD)]
    return video_transforms.Compose(ops)


transform     = build_transform(augment=False)
transform_aug = build_transform(augment=True)
print("Transforms ready.")

## 5. Streaming dataset

Streams directly from HuggingFace via webdataset.
GOPEN_CURL env variable handles auth for curl subprocesses.

In [ ]:
import tempfile, cv2


def shard_url(shard_idx):
    return (f"https://huggingface.co/datasets/{HF_DATASET}/"
            f"resolve/main/data/train-{shard_idx:05d}.tar")


def decode_mp4(mp4_bytes, tfm, n_frames=FRAMES_PER_CLIP):
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as f:
        f.write(mp4_bytes)
        tmp = f.name
    try:
        cap   = cv2.VideoCapture(tmp)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total < 2:
            return None
        indices = np.linspace(0, total - 1, n_frames, dtype=int)
        frames  = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if not ret:
                return None
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        video   = np.stack(frames)
        video_t = torch.from_numpy(video).permute(0, 3, 1, 2)
        return tfm(video_t)
    except Exception:
        return None
    finally:
        os.unlink(tmp)


class StreamDataset(IterableDataset):
    """
    Streams clips directly from HuggingFace using webdataset.
    GOPEN_CURL env variable handles auth.
    """
    def __init__(self, shard_range, max_clips=-1, augment=False):
        self.shard_range = shard_range
        self.max_clips   = max_clips
        self.tfm         = transform_aug if augment else transform

    def __iter__(self):
        count = 0
        start, end = self.shard_range
        urls = [shard_url(i) for i in range(start, end + 1)]
        dataset = wds.WebDataset(urls, shardshuffle=False,
                                  handler=wds.warn_and_continue)
        for sample in dataset:
            if self.max_clips > 0 and count >= self.max_clips:
                return
            mp4 = sample.get("mp4")
            if mp4 is None:
                continue
            tensor = decode_mp4(mp4, self.tfm)
            if tensor is None:
                continue
            count += 1
            if count % 200 == 0:
                print(f"  [data] {count} clips streamed", flush=True)
            yield tensor


def make_loader(shard_range, max_clips=-1, augment=False):
    ds = StreamDataset(shard_range, max_clips=max_clips, augment=augment)
    return DataLoader(ds, batch_size=BATCH_SIZE,
                      num_workers=4, pin_memory=True)


print("Streaming dataset helpers ready.")

## 6. Streaming sanity check

Verify streaming works before spending compute on training.

In [ ]:
print("Running streaming sanity check...")
t0 = time.time()
count = 0
for batch in make_loader((0, 0), max_clips=BATCH_SIZE * 2):
    print(f"  Batch shape: {batch.shape}")
    count += batch.shape[0]
    if count >= BATCH_SIZE * 2:
        break
print(f"  Streamed {count} clips in {time.time()-t0:.1f}s")
print("Sanity check passed. Streaming works.")

## 7. JEPA loss + masking

In [ ]:
def sample_masks(n_tokens, device):
    target     = torch.zeros(n_tokens, dtype=torch.bool, device=device)
    block_size = max(1, int(TARGET_MASK_RATIO * n_tokens / NUM_MASK_BLOCKS))
    for _ in range(NUM_MASK_BLOCKS):
        start = torch.randint(0, max(1, n_tokens - block_size), (1,)).item()
        target[start: start + block_size] = True
    context = ~target
    if context.sum() == 0:
        context[0] = True; target[0] = False
    return context, target


def jepa_loss(student, teacher, predictor, videos, device):
    with torch.no_grad():
        all_tok = teacher(videos)
    N = all_tok.shape[1]
    ctx_mask, tgt_mask = sample_masks(N, device)
    teacher_tgt        = all_tok[:, tgt_mask, :].mean(dim=1).detach()
    student_tok        = student(videos)
    prediction         = predictor(student_tok[:, ctx_mask, :])
    return F.mse_loss(prediction, teacher_tgt)


print("JEPA loss ready.")

## 8. Evaluation: Cycle@K and Overlap@K

In [ ]:
@torch.no_grad()
def extract_embeddings(encoder, shard_range, max_clips=VAL_CLIPS):
    encoder.eval()
    embs, sids = [], []
    start, end = shard_range
    per_shard  = max(1, max_clips // (end - start + 1))
    for shard_idx in range(start, end + 1):
        for batch in make_loader((shard_idx, shard_idx), max_clips=per_shard):
            feats = encoder(batch.to(DEVICE)).mean(dim=1)
            feats = F.normalize(feats, dim=-1)
            embs.append(feats.cpu())
            sids.extend([shard_idx] * batch.shape[0])
        if len(sids) >= max_clips:
            break
    embs = torch.cat(embs, dim=0)[:max_clips]
    sids = torch.tensor(sids[:max_clips])
    return embs, sids


def cycle_at_k(embs, sids, k=EVAL_K):
    N   = embs.shape[0]
    sim = embs @ embs.T
    sim.fill_diagonal_(-1e9)
    same = (sids.unsqueeze(0) == sids.unsqueeze(1))
    sim[same] = -1e9
    top1_b = sim.argmax(dim=1)
    topk_b = sim[top1_b].topk(k, dim=1).indices
    hits   = (topk_b == torch.arange(N).unsqueeze(1)).any(dim=1)
    return hits.float().mean().item()


def overlap_at_k(encoder, shard_range, max_clips=VAL_CLIPS, k=EVAL_K):
    encoder.eval()
    views = []
    for _ in range(2):
        e = []
        for batch in make_loader(shard_range, max_clips=max_clips, augment=True):
            with torch.no_grad():
                feats = encoder(batch.to(DEVICE)).mean(dim=1)
                feats = F.normalize(feats, dim=-1)
                e.append(feats.cpu())
            if sum(x.shape[0] for x in e) >= max_clips:
                break
        views.append(torch.cat(e)[:max_clips])
    n      = min(views[0].shape[0], views[1].shape[0])
    e1, e2 = views[0][:n], views[1][:n]

    def topk(e):
        s = e @ e.T; s.fill_diagonal_(-1e9)
        return s.topk(k, dim=1).indices

    tk1, tk2 = topk(e1), topk(e2)
    return float(np.mean([
        len(set(tk1[i].tolist()) & set(tk2[i].tolist())) / k
        for i in range(n)
    ]))


def evaluate(encoder):
    print("  [eval] extracting embeddings...", flush=True)
    t0         = time.time()
    embs, sids = extract_embeddings(encoder, VAL_SHARDS)
    cycle      = cycle_at_k(embs, sids)
    print(f"  [eval] Cycle@{EVAL_K}={cycle:.3f}, computing Overlap...", flush=True)
    overlap    = overlap_at_k(encoder, VAL_SHARDS)
    elapsed    = time.time() - t0
    print(f"  [eval] Overlap@{EVAL_K}={overlap:.3f}  ({elapsed:.0f}s)", flush=True)
    return {"cycle_at_k": cycle, "overlap_at_k": overlap}


print("Evaluation helpers ready.")

## 9. Shared utilities: EMA, checkpointing

In [ ]:
def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False


@torch.no_grad()
def ema_update(student, teacher, momentum=EMA_MOMENTUM):
    for ps, pt in zip(student.parameters(), teacher.parameters()):
        pt.data.mul_(momentum).add_((1 - momentum) * ps.data)


def save_ckpt(state, path):
    torch.save(state, path)
    print(f"  [ckpt] saved → {path}", flush=True)


def load_ckpt(path, student, teacher, predictor, optimizer=None):
    ckpt = torch.load(path, map_location="cpu")
    student.load_state_dict(ckpt["student"])
    teacher.load_state_dict(ckpt["teacher"])
    predictor.load_state_dict(ckpt["predictor"])
    if optimizer and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    return ckpt.get("session", 0), ckpt.get("epoch", 0), ckpt.get("metrics", {})


print("Utilities ready.")

## 10. SEEKR helpers: block outputs + forgettability scoring

In [ ]:
def get_block_outputs(encoder, videos):
    """Returns {block_idx: tensor [B, N, D]} via forward hooks on all blocks."""
    outputs = {}
    hooks   = []

    def make_hook(i):
        def h(m, inp, out):
            outputs[i] = out[0] if isinstance(out, tuple) else out
        return h

    for i, blk in enumerate(encoder.blocks):
        hooks.append(blk.register_forward_hook(make_hook(i)))
    _ = encoder(videos)
    for h in hooks:
        h.remove()
    return outputs


def top_k_forgettable_blocks(frozen_snap, student, rb_videos,
                              k=SEEKR_TOP_K_BLOCKS):
    """
    Forgettability score per block = mean L2 distance between
    frozen session snapshot and current student block outputs.
    Returns sorted list of block indices (most forgettable first).
    """
    with torch.no_grad():
        t_out = get_block_outputs(frozen_snap, rb_videos)
        s_out = get_block_outputs(student,     rb_videos)
    scores = {
        i: (s_out[i] - t_out[i]).norm(dim=-1).mean().item()
        for i in t_out if i in s_out
    }
    return sorted(scores, key=scores.get, reverse=True)[:k]


def seekr_extra_loss(student, teacher, predictor, videos, device,
                     replay_buffer, frozen_snap):
    """
    SEEKR additional loss:
      SEEKR_REPLAY_W  * JEPA loss on replay buffer
      SEEKR_DISTILL_W * selective block distillation on replay buffer
    """
    if not replay_buffer or frozen_snap is None:
        return torch.tensor(0.0, device=device)

    rb = torch.stack(replay_buffer).to(device)

    # Replay JEPA loss with gradient checkpointing to save memory
    replay_loss = torch.utils.checkpoint.checkpoint(
        lambda: jepa_loss(student, teacher, predictor, rb, device),
        use_reentrant=False
    )

    # Selective distillation on top-K forgettable blocks
    top_blocks = top_k_forgettable_blocks(frozen_snap, student, rb)
    with torch.no_grad():
        t_out = get_block_outputs(frozen_snap, rb)
    s_out   = get_block_outputs(student, rb)
    distill = sum(
        F.mse_loss(s_out[b], t_out[b].detach()) for b in top_blocks
    )

    return SEEKR_REPLAY_W * replay_loss + SEEKR_DISTILL_W * distill


print("SEEKR helpers ready.")

## 11. Experiment — SEEKR

Full encoder trains. Each step: JEPA loss on current batch + replay JEPA
loss + selective block distillation on most forgettable blocks.

In [ ]:
print("=" * 60)
print("SEEKR — Replay + Selective Distillation (JEPA loss)")
print("=" * 60)

# ── Build models ──────────────────────────────────────────────────────────────
student   = build_encoder().to(DEVICE)
teacher   = build_encoder().to(DEVICE)
teacher.load_state_dict(student.state_dict())
freeze_all(teacher)
predictor = JEPAPredictor(student.embed_dim).to(DEVICE)

total_params     = sum(p.numel() for p in student.parameters())
trainable_params = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f"  Trainable: {trainable_params:,} / {total_params:,} "
      f"({100*trainable_params/total_params:.2f}%)")

optimizer = torch.optim.AdamW([
    {"params": student.parameters(),   "lr": LR_BACKBONE},
    {"params": predictor.parameters(), "lr": LR_PREDICTOR},
])
scaler  = torch.amp.GradScaler('cuda')
metrics = {}

replay_buffer = []   # list of CPU tensors
frozen_snap   = None # frozen copy of student from end of prev session

# ── Session loop ──────────────────────────────────────────────────────────────
for t, shard_range in enumerate(SESSION_SHARDS):
    print(f"\n  Session {t+1}/{len(SESSION_SHARDS)}", flush=True)
    loader       = make_loader(shard_range, max_clips=CLIPS_PER_SESSION)
    epoch_losses = []
    student.train(); predictor.train()

    for epoch in range(NUM_EPOCHS):
        total, n = 0.0, 0
        for videos in loader:
            videos = videos.to(DEVICE)
            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                loss = jepa_loss(student, teacher, predictor, videos, DEVICE)
                loss = loss + seekr_extra_loss(
                    student, teacher, predictor, videos, DEVICE,
                    replay_buffer, frozen_snap
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(
                list(student.parameters()) + list(predictor.parameters()),
                GRAD_CLIP
            )
            scaler.step(optimizer)
            scaler.update()
            ema_update(student, teacher)
            total += loss.item(); n += 1

        ep = total / max(n, 1)
        epoch_losses.append(ep)
        print(f"    epoch {epoch+1}/{NUM_EPOCHS} | loss {ep:.4f}", flush=True)
        save_ckpt(
            {"student":   student.state_dict(),
             "teacher":   teacher.state_dict(),
             "predictor": predictor.state_dict(),
             "optimizer": optimizer.state_dict(),
             "session":   t,
             "epoch":     epoch,
             "metrics":   metrics},
            os.path.join(CKPT_DIR, f"seekr_session{t}_epoch{epoch}.pt")
        )

    # Fill replay buffer from this session
    print(f"  Filling replay buffer...", flush=True)
    n_new     = REPLAY_BUDGET // len(SESSION_SHARDS)
    new_clips = []
    for batch in make_loader(shard_range, max_clips=n_new * BATCH_SIZE):
        for clip in batch:
            new_clips.append(clip.cpu())
            if len(new_clips) >= n_new:
                break
        if len(new_clips) >= n_new:
            break
    replay_buffer = (replay_buffer + new_clips)[-REPLAY_BUDGET:]
    print(f"  Replay buffer: {len(replay_buffer)} clips", flush=True)

    # Snapshot student as frozen reference for next session
    frozen_snap = copy.deepcopy(student).eval()
    freeze_all(frozen_snap)

    res = evaluate(student)
    metrics[t] = {"losses": epoch_losses, **res}
    print(f"  Session {t+1} → Cycle@{EVAL_K}={res['cycle_at_k']:.3f}  "
          f"Overlap@{EVAL_K}={res['overlap_at_k']:.3f}", flush=True)

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)

## 12. Results summary

In [ ]:
print("\nSEEKR — Session-by-session retrieval metrics:")
print(f"{'Session':<10} {'Cycle@'+str(EVAL_K):<15} {'Overlap@'+str(EVAL_K):<15} {'Final loss':<12}")
print("-" * 52)
for t, m in metrics.items():
    overlap_str = f"{m['overlap_at_k']:.3f}" if m['overlap_at_k'] is not None else "N/A"
    print(f"{t+1:<10} {m['cycle_at_k']:<15.3f} "
          f"{overlap_str:<15} "
          f"{m['losses'][-1]:<12.4f}")

if len(metrics) > 1:
    bwt = metrics[len(metrics)-1]["cycle_at_k"] - metrics[0]["cycle_at_k"]
    print(f"\nBWT (Cycle@{EVAL_K}): {bwt:+.3f}")
    print("(negative = forgetting, 0 = none)")
    print("\nComparison (Colab small-scale, 150 clips/session):")
    print("  SSIAT/SAPT BWT: +0.000  |  SAFE BWT: +0.000")

## 13. Download checkpoints

Before destroying the instance, download checkpoints via Jupyter file browser
or run this cell to zip them up.

In [ ]:
import glob

# List all saved checkpoints
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "seekr_*.pt")))
print(f"Saved checkpoints ({len(ckpts)}):")
for c in ckpts:
    size = os.path.getsize(c) / 1e6
    print(f"  {os.path.basename(c)}  ({size:.0f} MB)")

# Zip for easy download
os.system(f"zip -j /workspace/seekr_checkpoints.zip {CKPT_DIR}/seekr_*.pt")
print("\nZipped → /workspace/seekr_checkpoints.zip")
print("Download via Jupyter file browser (left panel → right click → download)")

## 14. Resume helper

In [ ]:
# If kernel dies mid-run:
# 1. Re-run cells 0-10
# 2. Run this cell to reload latest checkpoint
# 3. In Cell 11, change SESSION_SHARDS loop to start from last_session

ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "seekr_*.pt")))
if ckpts:
    latest = ckpts[-1]
    print(f"Latest checkpoint: {latest}")
    last_session, last_epoch, saved_metrics = load_ckpt(
        latest, student, teacher, predictor, optimizer)
    metrics = saved_metrics
    print(f"Resumed: session={last_session}, epoch={last_epoch}")
    print(f"Saved metrics: {list(saved_metrics.keys())}")
else:
    print("No checkpoints found.")